# BdSLW60 - Week 2, session 2: second seed + masked-pooling ablation


In [ ]:
# ---- Cell 1: config and paths ----
import os, re, json, time, math, random, collections, subprocess, sys, shutil, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore', message='undefined skeleton graph')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='torch.nn.init')

SESSION_START = time.time()        # the 12-hour Kaggle limit counts from here (roughly)
WORK = Path('/kaggle/working')
CACHE = WORK / 'bdslw60_w2'
KP_DIR = CACHE / 'keypoints'
for d in (CACHE, KP_DIR):
    d.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path('/kaggle/input')

# Week-1 keypoints: found automatically (a folder named 'keypoints' holding U*W*.npz files). 
KP_SOURCE = "/kaggle/input/notebooks/rishta19/bdslw60-week-1-v2/bdslw60_cache/keypoints"
if KP_SOURCE is None:
    cands = sorted({p.parent for p in INPUT_ROOT.rglob('U*W*.npz') if p.parent.name == 'keypoints'},
                   key=lambda d: -len(list(d.glob('*.npz'))))
    KP_SOURCE = cands[0] if cands else None
print('week-1 keypoints:', KP_SOURCE, '(%d files)' % len(list(Path(KP_SOURCE).glob('*.npz'))) if KP_SOURCE else '')
assert KP_SOURCE is not None, 'Add the week-1 v2 notebook output as an input (it holds bdslw60_cache/keypoints).'

T_OUT = 64
N_NODES = 49
MAX_GAP = 5
TEST_SIGNERS = (4, 8)            # never evaluated before week 3
P1_VAL_SIGNER = 5
INNER_VAL_SIGNERS = (1, 10, 13)
MIN_VAL_CLIPS = 500
SLGCN_REPO = 'https://github.com/jackyjsy/CVPR21Chal-SLR.git'
SLGCN_COMMIT = 'de6c53a6fbbc08a5d9a728e33614a4541836714c'   # pinned so results are reproducible

WORD_IDS = (list(range(1, 13)) + [19, 20] + list(range(37, 51)) + list(range(91, 101))
            + [111, 112] + list(range(211, 221)) + list(range(351, 361)))
assert len(WORD_IDS) == 60
WORD2CLS = {w: i for i, w in enumerate(WORD_IDS)}
TABLE6 = {1: 975, 2: 744, 3: 583, 4: 637, 5: 600, 6: 815, 7: 107, 8: 639, 9: 656,
          10: 383, 11: 771, 12: 655, 13: 621, 14: 112, 15: 655, 16: 97, 17: 116, 18: 141}
VID_RE = re.compile(r'U(\d+)W(\d+)([A-Za-z]*)', re.I)
all_files = [p for p in INPUT_ROOT.rglob('*') if p.is_file() and 'bdslw60' in str(p).lower()]
print('config ok')


week-1 keypoints: /kaggle/input/notebooks/rishta19/bdslw60-week-1-v2/bdslw60_cache/keypoints (770 files)
config ok


In [2]:
# ---- Cell 2: official SL-GCN code (pinned commit) + the 49-node BdSL graph file ----
REPO_DIR = WORK / 'CVPR21Chal-SLR'
if not (REPO_DIR / 'SL-GCN').exists():
    subprocess.run(['git', 'clone', '-q', SLGCN_REPO, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-q', SLGCN_COMMIT], check=True)
SLGCN_DIR = REPO_DIR / 'SL-GCN'

# BdSL graph: 7 body nodes (nose, shoulders, elbows, wrists) + 21 left-hand + 21 right-hand nodes.
HAND_EDGES = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12),
              (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]
BODY_EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 5), (4, 6)]
EDGES = (BODY_EDGES + [(a + 7, b + 7) for a, b in HAND_EDGES] + [(a + 28, b + 28) for a, b in HAND_EDGES]
         + [(5, 7), (6, 28)])

def build_parents(edges, V=N_NODES, root=0):
    adj = collections.defaultdict(list)
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    parent = [-1] * V
    parent[root] = root
    queue = [root]
    while queue:
        u = queue.pop(0)
        for w in adj[u]:
            if parent[w] == -1:
                parent[w] = u
                queue.append(w)
    assert all(p >= 0 for p in parent), 'graph is not connected'
    return np.array(parent)

PARENT = build_parents(EDGES)
INWARD = [(int(v), int(PARENT[v])) for v in range(N_NODES) if PARENT[v] != v]   # child -> parent, as in sign_27.py
GRAPH_FILE = """import sys
sys.path.extend(['../'])
from graph import tools

num_node = %d
self_link = [(i, i) for i in range(num_node)]
inward = %r
outward = [(j, i) for (i, j) in inward]
neighbor = inward + outward


class Graph:
    def __init__(self, labeling_mode='spatial'):
        self.num_node = num_node
        self.self_link = self_link
        self.inward = inward
        self.outward = outward
        self.neighbor = neighbor
        self.A = tools.get_spatial_graph(num_node, self_link, inward, outward)
""" % (N_NODES, INWARD)
(SLGCN_DIR / 'graph' / 'bdsl49.py').write_text(GRAPH_FILE)

if str(SLGCN_DIR) not in sys.path:
    sys.path.insert(0, str(SLGCN_DIR))
from model.decouple_gcn_attn import Model as SLGCN, TCN_GCN_unit, unit_gcn
from graph.bdsl49 import Graph as BdslGraph
GRAPH_A = BdslGraph('spatial').A                   # (3, V, V): self, inward, outward
assert GRAPH_A.shape == (3, N_NODES, N_NODES)
print('SL-GCN code loaded from', SLGCN_DIR, '| commit', SLGCN_COMMIT[:8], '| graph edges:', len(INWARD))


SL-GCN code loaded from /kaggle/working/CVPR21Chal-SLR/SL-GCN | commit de6c53a6 | graph edges: 48


In [3]:
# ---- Cell 3: videos + annotations -> one table of trials (same parser as week 1, plus the FL fix) ----
rows = []
for p in all_files:
    if p.suffix.lower() == '.mp4':
        m = VID_RE.search(p.stem)
        if m:
            rows.append(dict(path=str(p), stem=p.stem, user=int(m[1]), word=int(m[2]), view=m[3].upper()))
vids_all = pd.DataFrame(rows)
# FIX 3: when several front videos share a (user, word) (e.g. U1W357F_1, _2, _3), use the one week 1 used
# (its keypoints are in KP_SOURCE), so the annotated frame numbers refer to the right recording.
vids_all['in_week1'] = vids_all.stem.map(lambda s: (Path(KP_SOURCE) / (s + '.npz')).exists())
vids_all = vids_all.sort_values(['in_week1', 'stem'], ascending=[False, True]).reset_index(drop=True)
for (u, w), g in vids_all[vids_all.view == 'F'].groupby(['user', 'word']):
    if len(g) > 1:
        print('U%dW%d has %d front videos: using %s, not %s' % (u, w, len(g), g.stem.iloc[0], g.stem.iloc[1:].tolist()))
front = vids_all[vids_all.view == 'F'].drop_duplicates(['user', 'word'])
# FIX 1: a non-front video (e.g. U18W358FL) is used only when that (user, word) has no front-view video at all
have_front = set(zip(front.user, front.word))
extra = vids_all[(vids_all.view != 'F') & ~vids_all.apply(lambda r: (r.user, r.word) in have_front, axis=1)]
extra = extra.drop_duplicates(['user', 'word'])
for r in extra.itertuples():
    print('using non-front video because no front copy exists:', r.stem)
for r in vids_all[(vids_all.view != 'F') & ~vids_all.stem.isin(extra.stem)].itertuples():
    print('not used (a front copy exists):', r.stem)
videos = pd.concat([front, extra]).reset_index(drop=True)
videos = videos[videos.word.isin(WORD_IDS)].reset_index(drop=True)
USE_VIEWS = {(r.user, r.word): r.view for r in videos.itertuples()}
print('videos used:', len(videos))

json_files = sorted(p for p in all_files if p.suffix.lower() == '.json' and 'output' in p.name.lower())

def iter_dicts(obj, path=()):
    if isinstance(obj, dict):
        yield path, obj
        for k, v in obj.items():
            yield from iter_dicts(v, path + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_dicts(v, path + ('[%d]' % i,))

def as_int(v):
    try:
        return int(float(v))
    except Exception:
        return None

segs = []
for jp in json_files:
    with open(jp, encoding='utf-8') as f:
        data = json.load(f)
    for path, d in iter_dicts(data):
        if 'starting' not in d or 'ending' not in d:
            continue
        sig = ' '.join(list(path) + [str(v) for v in d.values() if isinstance(v, (str, int))])
        m = VID_RE.search(sig)
        if m:
            user, word, view = int(m[1]), int(m[2]), m[3].upper()
        else:                                             # fallback: user and word from the tree levels (W1 -> U1 -> ...)
            pu = [int(x[1:]) for x in path if re.fullmatch(r'U\d+', x)]
            pw = [int(x[1:]) for x in path if re.fullmatch(r'W\d+', x)]
            if not pu or not pw:
                continue
            user, word = pu[-1], pw[-1]
            view = USE_VIEWS.get((user, word), '')
        # week 1 kept only view '' or 'F'; now any view is kept if it is the video we use for this (user, word)
        if view not in ('', 'F') and USE_VIEWS.get((user, word)) != view:
            continue
        hand = next((p for p in path if p in ('LeftHand', 'RightHand')), None)
        starts, ends = d['starting'], d['ending']
        if not isinstance(starts, list):
            starts, ends = [starts], [ends]
        for s, e in zip(starts, ends):
            s, e = as_int(s), as_int(e)
            if s is None or e is None or e <= s:
                continue
            segs.append(dict(user=user, word=word, start=s, end=e, hand='L' if hand == 'LeftHand' else 'R', src=jp.name))
segs = pd.DataFrame(segs).drop_duplicates(['user', 'word', 'start', 'end'])
assert segs.hand.notna().all()
trials = segs.merge(videos[['user', 'word', 'stem', 'path']], on=['user', 'word'], how='inner')
trials['cls'] = trials.word.map(WORD2CLS)
trials = trials.sort_values(['stem', 'start']).reset_index(drop=True)
cmp = pd.DataFrame({'ours': trials.groupby('user').size(), 'paper': pd.Series(TABLE6)}).fillna(0).astype(int)
cmp['diff'] = cmp.ours - cmp.paper
print('trials:', len(trials), '(paper 9307; week 1 had 9297)')
print(cmp.T.to_string())


U1W357 has 3 front videos: using U1W357F_2, not ['U1W357F_1', 'U1W357F_3']
using non-front video because no front copy exists: U18W358FL
videos used: 775
trials: 9300 (paper 9307; week 1 had 9297)
        1    2    3    4    5    6    7    8    9    10   11   12   13   14   15  16   17   18
ours   970  742  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
paper  975  744  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
diff    -5   -2    0    0    0    0    0    0    0    0    0    0    0    0    0   0    0    0


In [4]:
# ---- Cell 4: keypoints - copy the week-1 files, extract only what is missing (e.g. the FL video) ----
need = trials.groupby('stem').agg(path=('path', 'first'), f0=('start', 'min'), f1=('end', 'max')).reset_index()
n_copied = 0
for r in need.itertuples():
    src, dst = Path(KP_SOURCE) / (r.stem + '.npz'), KP_DIR / (r.stem + '.npz')
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        n_copied += 1
todo = [r for r in need.itertuples() if not (KP_DIR / (r.stem + '.npz')).exists()]
print('copied %d keypoint files; %d video(s) still to extract: %s' % (n_copied, len(todo), [r.stem for r in todo]))

if todo:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe'], check=True)
    import urllib.request, cv2
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    model_path = WORK / 'holistic_landmarker.task'
    if not model_path.exists():
        found = list(INPUT_ROOT.rglob('holistic_landmarker.task'))
        if found:
            shutil.copy(found[0], model_path)
        else:
            urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/holistic_landmarker/'
                                       'holistic_landmarker/float16/latest/holistic_landmarker.task', model_path)
    POSE_IDX = [0, 11, 12, 13, 14, 15, 16]
    for r in todo:                                   # same settings as the week-1 extractor
        f0, f1 = max(0, int(r.f0) - 5), int(r.f1) + 5
        opts = mp_vision.HolisticLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=str(model_path)),
            running_mode=mp_vision.RunningMode.VIDEO,
            min_pose_detection_confidence=0.5, min_pose_landmarks_confidence=0.5,
            min_hand_landmarks_confidence=0.5, min_face_detection_confidence=0.5,
            min_face_landmarks_confidence=0.5)
        h = mp_vision.HolisticLandmarker.create_from_options(opts)
        cap = cv2.VideoCapture(r.path)
        for _ in range(f0):
            if not cap.grab():
                break
        coords, masks = [], []
        for fi in range(f0, f1 + 1):
            ok, frame = cap.read()
            if not ok:
                break
            H, W = frame.shape[:2]
            if max(H, W) > 640:
                s = 640.0 / max(H, W)
                frame = cv2.resize(frame, (int(W * s), int(H * s)))
            rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            res = h.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), (fi - f0) * 33)
            c = np.zeros((49, 3), np.float32); m = np.zeros(49, np.uint8)
            if res.pose_landmarks:
                for k, idx in enumerate(POSE_IDX):
                    lm = res.pose_landmarks[idx]
                    c[k] = (lm.x, lm.y, lm.z); m[k] = 1
            for hand, sl, wrist in ((res.left_hand_landmarks, slice(7, 28), 5), (res.right_hand_landmarks, slice(28, 49), 6)):
                if hand:
                    arr = np.array([[p.x, p.y, p.z] for p in hand], np.float32)
                    if m[wrist]:
                        arr[:, 2] += c[wrist, 2]
                    c[sl] = arr; m[sl] = 1
            coords.append(c); masks.append(m)
        h.close(); cap.release()
        np.savez_compressed(KP_DIR / (r.stem + '.npz'), coords=np.stack(coords), mask=np.stack(masks), f0=f0)
        print('extracted', r.stem, len(coords), 'frames')
print('keypoint files ready:', len(list(KP_DIR.glob('*.npz'))))


copied 770 keypoint files; 1 video(s) still to extract: ['U18W358FL']


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1793131553.392385     104 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793131553.425160     104 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793131553.430194     105 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793131553.432361     103 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature infer

extracted U18W358FL 632 frames
keypoint files ready: 771


In [5]:
# ---- Cell 5: preprocessing functions + self-tests on synthetic data ----
LEFT_BODY, RIGHT_BODY = [1, 3, 5], [2, 4, 6]
PERM = np.arange(N_NODES)
for l, r in zip(LEFT_BODY, RIGHT_BODY):
    PERM[l], PERM[r] = r, l
PERM[7:28], PERM[28:49] = np.arange(28, 49), np.arange(7, 28)

def fill_gaps(coords, mask, max_gap=MAX_GAP):
    # linear interpolation of gaps up to max_gap frames that are bounded by detections on both sides
    coords, mask = coords.copy(), mask.copy()
    for n in range(coords.shape[1]):
        idx = np.flatnonzero(mask[:, n])
        for a, b in zip(idx[:-1], idx[1:]):
            gap = b - a - 1
            if 1 <= gap <= max_gap:
                w = ((np.arange(a + 1, b) - a) / (b - a))[:, None]
                coords[a + 1:b, n] = (1 - w) * coords[a, n] + w * coords[b, n]
                mask[a + 1:b, n] = 1
    return coords, mask

def normalize(coords, mask):
    # centre on mid-shoulder, divide by shoulder width, per frame; missing nodes stay exactly 0
    ls, rs = coords[:, 1], coords[:, 2]
    ok = (mask[:, 1] > 0) & (mask[:, 2] > 0)
    if not ok.any():
        return None, None
    mid = (ls + rs) / 2
    width = np.linalg.norm((ls - rs)[:, :2], axis=1)
    good = ok & (width > 1e-4)
    if not good.any():
        return None, None
    mid_f = np.where(ok[:, None], mid, np.median(mid[ok], axis=0))
    width_f = np.where(good, width, np.median(width[good]))
    out = (coords - mid_f[:, None, :]) / width_f[:, None, None]
    out = out * mask[..., None]
    return out.astype(np.float32), mask

def mirror(coords, mask):
    c = coords[:, PERM].copy()
    c[..., 0] *= -1
    return c, mask[:, PERM]

def resample(coords, mask, t_out=T_OUT):
    # mask-weighted linear interpolation so a detected frame never blends with a zero-filled neighbour
    T = len(coords)
    if T == 1:
        return np.repeat(coords, t_out, 0), np.repeat(mask, t_out, 0)
    src = np.linspace(0, T - 1, t_out)
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, T - 1)
    w = (src - lo)
    wv, wc = w[:, None], w[:, None, None]
    mf = mask.astype(np.float32)
    num = (1 - wc) * (coords[lo] * mf[lo][..., None]) + wc * (coords[hi] * mf[hi][..., None])
    den = (1 - wv) * mf[lo] + wv * mf[hi]
    mask_out = mask[np.rint(src).astype(int)]
    coords_out = np.where(mask_out[..., None] > 0, num / np.maximum(den[..., None], 1e-6), 0.0)
    return coords_out.astype(np.float32), mask_out

# ---- self-tests ----
rng = np.random.default_rng(0)
c = rng.normal(size=(30, N_NODES, 3)).astype(np.float32)
c[:, 1] = [0.6, 0.5, 0.0]; c[:, 2] = [0.4, 0.5, 0.0]          # shoulders: width 0.2
m = np.ones((30, N_NODES), np.uint8)
m[5:8, 10] = 0; m[12:20, 11] = 0; m[0:3, 12] = 0              # short gap, long gap, leading gap
c = c * m[..., None]
c2, m2 = fill_gaps(c, m)
assert m2[5:8, 10].all() and not m2[12:20, 11].any() and not m2[0:3, 12].any()
n, nm = normalize(c2, m2)
assert np.allclose(np.abs(n[:, 1, 0]), 0.5, atol=1e-4)      # shoulders end up at +-0.5
assert np.all(n[nm == 0] == 0)
mc, mm = mirror(n, nm)
mc2, mm2 = mirror(mc, mm)
assert np.allclose(mc2, n) and (mm2 == nm).all()
assert np.allclose(mc[:, 1, 0], -n[:, 2, 0])                 # left/right shoulders swapped and mirrored
const = np.ones((20, N_NODES, 3), np.float32); cm = np.ones((20, N_NODES), np.uint8)
cm[8:14, 20] = 0; const = const * cm[..., None]
rc, rm = resample(const, cm)
assert rc.shape == (T_OUT, N_NODES, 3) and rm.shape == (T_OUT, N_NODES)
assert np.allclose(rc[rm[:, 20] > 0, 20], 1.0)               # no partial values at mask borders
assert np.all(rc[rm == 0] == 0)
print('preprocessing self-tests passed')

preprocessing self-tests passed


In [6]:
# ---- Cell 6: build the arrays (B0 inputs: zero-filled + detection mask), drop duplicate trials ----
def build(trials):
    n = len(trials)
    X = np.zeros((n, 3, T_OUT, N_NODES), np.float32)
    M = np.zeros((n, T_OUT, N_NODES), np.uint8)
    ok = np.zeros(n, bool)
    hand_missing_raw = np.full(n, np.nan, np.float32)
    cur_stem, kp = None, None
    for i, r in enumerate(trials.itertuples()):
        if r.stem != cur_stem:
            f = KP_DIR / (r.stem + '.npz')
            kp = None
            if f.exists():
                z = np.load(f)
                kp = dict(coords=z['coords'], mask=z['mask'], f0=int(z['f0']))
            cur_stem = r.stem
        if kp is None:
            continue
        s, e = r.start - kp['f0'], r.end - kp['f0']
        if s < 0 or e >= len(kp['coords']) or e - s < 2:
            continue
        c, m = kp['coords'][s:e + 1], kp['mask'][s:e + 1]
        hand_missing_raw[i] = (m[:, 7:] == 0).mean()          # before any processing, for the missing-rate analysis
        c, m = normalize(c, m)
        if c is None:
            continue
        if r.hand == 'L':
            c, m = mirror(c, m)
        c, m = resample(c, m)
        X[i] = c.transpose(2, 0, 1); M[i] = m; ok[i] = True
    return X, M, ok, hand_missing_raw

t0 = time.time()
X, M, ok, hm = build(trials)
meta = trials[ok].reset_index(drop=True)
X, M = X[ok], M[ok]
meta['hand_missing'] = hm[ok]
print('built in %.0f s | usable trials %d / %d' % (time.time() - t0, len(meta), len(trials)))

# FIX 2: identical trials within one signer (double annotations) - keep the first copy
h = [hashlib.md5(np.round(X[i], 4).tobytes()).hexdigest() for i in range(len(X))]
meta['hash'] = h
dup = meta.duplicated(['user', 'hash'], keep='first').values
print('identical duplicate trials removed:', int(dup.sum()))
for r in meta[dup].itertuples():
    print('   ', r.stem, r.start, r.end)
cross = meta[~dup].groupby('hash').user.nunique()
assert (cross <= 1).all(), 'identical trials under different signers - check before training'
X, M, meta = X[~dup], M[~dup], meta[~dup].reset_index(drop=True)
y = meta.cls.values.astype(np.int64)
np.save(CACHE / 'X.npy', X); np.save(CACHE / 'M.npy', M); np.save(CACHE / 'y.npy', y)
meta.to_csv(CACHE / 'meta.csv', index=False)
cmp = pd.DataFrame({'ours': meta.groupby('user').size(), 'paper': pd.Series(TABLE6)}).fillna(0).astype(int)
cmp['diff'] = cmp.ours - cmp.paper
print('\nfinal trials per signer vs Table 6:')
print(cmp.T.to_string())
print('total %d | test U4+U8 %d (paper 1276) | classes %d' % (len(meta), meta.user.isin(TEST_SIGNERS).sum(), len(set(y))))


built in 8 s | usable trials 9299 / 9300
identical duplicate trials removed: 0

final trials per signer vs Table 6:
        1    2    3    4    5    6    7    8    9    10   11   12   13   14   15  16   17   18
ours   969  742  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
paper  975  744  583  637  600  815  107  639  656  383  771  655  621  112  655  97  116  141
diff    -6   -2    0    0    0    0    0    0    0    0    0    0    0    0    0   0    0    0
total 9299 | test U4+U8 1276 (paper 1276) | classes 60


In [7]:
# ---- Cell 7: split files (P1 official split, inner model-selection folds, 18 leave-one-signer-out folds) ----
users = sorted(int(u) for u in meta.user.unique())
clips_per_user = meta.groupby('user').size().to_dict()
lh_per_user = meta[meta.hand == 'L'].groupby('user').size().reindex(users, fill_value=0).to_dict()
def idx_of(us):
    return np.flatnonzero(meta.user.isin(us).values)

# P1: final runs. Train 15 signers, pick the checkpoint on U5, test on U4+U8 once (week 3 only).
p1 = {'train': idx_of([u for u in users if u not in TEST_SIGNERS + (P1_VAL_SIGNER,)]),
      'val': idx_of([P1_VAL_SIGNER]), 'test': idx_of(list(TEST_SIGNERS))}
print('P1 sizes -> train %d | val (U%d) %d | test (U4+U8) %d   (paper/plan: ~7431 / 600 / 1276)'
      % (len(p1['train']), P1_VAL_SIGNER, len(p1['val']), len(p1['test'])))

# Inner folds: all model decisions (B0 vs B1 vs B2 ...). Each holds out one of INNER_VAL_SIGNERS and trains
# on every other non-test signer. U4/U8 appear in none of them.
inner = {}
for v in INNER_VAL_SIGNERS:
    tr_users = [u for u in users if u not in TEST_SIGNERS + (v,)]
    inner[v] = {'train': idx_of(tr_users), 'val': idx_of([v])}
    assert not set(meta.user.values[inner[v]['train']]) & set(TEST_SIGNERS + (v,))
    print('inner fold val U%-2d: train %d | val %d (left-handed %d)'
          % (v, len(inner[v]['train']), len(inner[v]['val']), lh_per_user[v]))

# LOSO: a TEST-TIME protocol (two of its folds test on U4 and U8). Run it in week 3 only.
# Each fold's validation signer comes from the signers with >= MIN_VAL_CLIPS clips.
big = [u for u in users if clips_per_user[u] >= MIN_VAL_CLIPS]
loso = {}
for k, s in enumerate(users):
    cands = [u for u in big if u != s]
    v = cands[k % len(cands)]
    loso[s] = {'train': idx_of([u for u in users if u not in (s, v)]), 'val': idx_of([v]), 'test': idx_of([s])}
print('LOSO validation signers (test -> val):', {s: int(meta.user.values[d['val'][0]]) for s, d in loso.items()})
assert all(len(d['val']) >= MIN_VAL_CLIPS for d in loso.values())

splits = {'P1': {k: v.tolist() for k, v in p1.items()},
          'INNER': {str(v): {k: a.tolist() for k, a in d.items()} for v, d in inner.items()},
          'LOSO': {str(s): {k: v.tolist() for k, v in d.items()} for s, d in loso.items()}}
json.dump(splits, open(CACHE / 'splits.json', 'w'))
print('LOSO folds written:', len(loso))
print('files in cache:', sorted(p.name for p in CACHE.iterdir() if p.is_file()))


P1 sizes -> train 7423 | val (U5) 600 | test (U4+U8) 1276   (paper/plan: ~7431 / 600 / 1276)
inner fold val U1 : train 7054 | val 969 (left-handed 648)
inner fold val U10: train 7640 | val 383 (left-handed 148)
inner fold val U13: train 7402 | val 621 (left-handed 186)
LOSO validation signers (test -> val): {1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 6: 8, 7: 8, 8: 11, 9: 12, 10: 12, 11: 15, 12: 1, 13: 2, 14: 2, 15: 4, 16: 4, 17: 5, 18: 6}
LOSO folds written: 18
files in cache: ['M.npy', 'X.npy', 'meta.csv', 'splits.json', 'y.npy']


In [8]:
# ---- Cell 8: augmentation and bone/motion streams (graph and PARENT come from cell 2) ----
def time_warp(x, m, gamma):
    # Monotonic warp that keeps the whole sign: frame t samples the clip at (T-1) * (t/(T-1))**gamma.
    # gamma < 1: the start is played slower and the end faster; gamma > 1: the opposite. First and last frames stay.
    C, T, V = x.shape
    src = (T - 1) * np.linspace(0, 1, T) ** gamma
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, T - 1)
    w = src - lo
    mf = m.astype(np.float32)
    num = (1 - w)[None, :, None] * x[:, lo] * mf[lo][None] + w[None, :, None] * x[:, hi] * mf[hi][None]
    den = (1 - w)[:, None] * mf[lo] + w[:, None] * mf[hi]
    m_out = m[np.rint(src).astype(int)]
    x_out = np.where(m_out[None] > 0, num / np.maximum(den[None], 1e-6), 0.0)
    return x_out.astype(np.float32), m_out

def augment(x, m, rng, p_rot=0.5, p_scale=0.5, p_speed=0.5, p_hdrop=0.2):
    # x: (C, T, V) coordinates, m: (T, V) detection mask. Training only.
    C, T, V = x.shape
    x, m = x.copy(), m.copy()
    if rng.random() < p_speed:                       # rhythm change; keeps the whole sign (see time_warp)
        x, m = time_warp(x, m, float(np.exp(rng.uniform(np.log(0.8), np.log(1.25)))))
    if rng.random() < p_rot:                         # rotation +-15 degrees in the image plane
        th = np.deg2rad(rng.uniform(-15, 15))
        c, s_ = np.cos(th), np.sin(th)
        x0, x1 = x[0].copy(), x[1].copy()
        x[0], x[1] = c * x0 - s_ * x1, s_ * x0 + c * x1
    if rng.random() < p_scale:                       # scale 0.8-1.2
        x = x * rng.uniform(0.8, 1.2)
    if rng.random() < p_hdrop:                       # one hand disappears for a short span
        sl = slice(7, 28) if rng.random() < 0.5 else slice(28, 49)
        L = int(rng.integers(8, 25))
        st = int(rng.integers(0, max(1, T - L)))
        x[:, st:st + L, sl] = 0
        m[st:st + L, sl] = 0
    return (x * m[None]).astype(np.float32), m

def make_stream(x, m, stream, use_mask, use_z=True):
    # returns features (C', T, V) and the matching mask (T, V)
    if stream == 'joint':
        f, fm = x, m
    elif stream == 'bone':
        f = x - x[:, :, PARENT]
        fm = m * m[:, PARENT]
    elif stream == 'joint_motion':
        f = np.zeros_like(x); fm = np.zeros_like(m)
        f[:, :-1] = x[:, 1:] - x[:, :-1]
        fm[:-1] = m[1:] * m[:-1]
    elif stream == 'bone_motion':
        b, bm = make_stream(x, m, 'bone', False)
        f = np.zeros_like(b); fm = np.zeros_like(bm)
        f[:, :-1] = b[:, 1:] - b[:, :-1]
        fm[:-1] = bm[1:] * bm[:-1]
    else:
        raise ValueError(stream)
    if use_mask:
        f = f * fm[None]
    if not use_z:
        f = f[:2]
    return f.astype(np.float32), fm

# ---- self-tests ----
assert PARENT[0] == 0 and len(set(PARENT.tolist())) > 10
_rng = np.random.default_rng(1)
_x = _rng.normal(size=(3, T_OUT, N_NODES)).astype(np.float32)
_m = (_rng.random((T_OUT, N_NODES)) > 0.3).astype(np.uint8)
_x = _x * _m[None]
for _s in ('joint', 'bone', 'joint_motion', 'bone_motion'):
    for _um in (False, True):
        _f, _fm = make_stream(_x, _m, _s, _um)
        assert _f.shape == (3, T_OUT, N_NODES) and _fm.shape == (T_OUT, N_NODES)
        if _um:
            assert np.all(_f[:, _fm == 0] == 0)
_w, _wm = time_warp(_x, _m, 0.8)
assert _w.shape == _x.shape and (_wm[0] == _m[0]).all() and (_wm[-1] == _m[-1]).all()   # whole sign kept
assert np.allclose(time_warp(_x, _m, 1.0)[0], _x)                                       # gamma 1 = identity
_a, _am = augment(_x, _m, np.random.default_rng(2), 1, 1, 1, 1)
assert _a.shape == _x.shape and np.all(_a[:, _am == 0] == 0)
print('augmentation and stream self-tests passed')

augmentation and stream self-tests passed


In [9]:
# ---- Cell 9: models - official SL-GCN (S0), + missing-aware blocks (S2), + signer-adversarial head (S3) ----
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True              # fixed input sizes: let cuDNN pick the fastest kernels

# fp32 islands: these parts overflow or lose accuracy in fp16, so they always run in fp32, even under autocast.
from model.dropSke import DropBlock_Ske
from model.dropT import DropBlockT_1d

def _autocast_on():
    try:
        return torch.is_autocast_enabled('cuda')
    except TypeError:                               # older torch
        return torch.is_autocast_enabled()

def fp32_island(fn):
    if getattr(fn, '_fp32_island', False):
        return fn
    def wrapped(self, x, *args, **kw):
        if torch.is_tensor(x) and x.is_cuda and _autocast_on():   # l1 passes a plain 0 (no residual)
            with torch.autocast(device_type='cuda', enabled=False):
                args = tuple(a.float() if torch.is_tensor(a) and a.is_floating_point() else a for a in args)
                return fn(self, x.float(), *args, **kw)   # every tensor input in fp32 (e.g. the mask)
        return fn(self, x, *args, **kw)
    wrapped._fp32_island = True
    return wrapped

for _cls in (DropBlock_Ske, DropBlockT_1d, unit_gcn):   # patched once; the official classes are otherwise unchanged
    _cls.forward = fp32_island(_cls.forward)
print('device:', device)
assert device == 'cuda', 'The official SL-GCN code creates some tensors on CUDA: turn on the GPU.'

A_BAR = torch.tensor(np.abs(GRAPH_A).sum(0), dtype=torch.float32)   # (V, V) fixed graph, self links included
A_HOP = (A_BAR > 0).float()

class MissingAwareGCN(nn.Module):
    """Wraps an official unit_gcn. Missing nodes (mask 0) are replaced by a learned token, send no messages to
    their neighbours (they still keep their own state through the self link), and each node's output is rescaled
    by the share of its graph neighbourhood that was observed (as in partial convolution)."""
    def __init__(self, g, in_channels):
        super().__init__()
        self.g = g
        self.token = nn.Parameter(torch.zeros(1, in_channels, 1, N_NODES))
        self.register_buffer('A_bar', A_BAR.clone())

    def forward(self, x0, m):                             # x0: (N, C, T, V), m: (N, T, V) float in {0, 1}
        g = self.g
        mm = m[:, None]                                   # (N, 1, T, V)
        xt = x0 * mm + (1 - mm) * self.token
        learn_A = g.DecoupleA.repeat(1, g.out_channels // g.groups, 1, 1)
        A = torch.cat([g.norm(learn_A[k:k + 1]) for k in range(3)], 0)          # (3, C', V, V), source v -> target w
        x = torch.einsum('nctw,cd->ndtw', xt, g.Linear_weight).contiguous() + g.Linear_bias
        x = g.bn0(x)
        n, kc, t, v = x.size()
        x = x.view(n, g.num_subset, kc // g.num_subset, t, v)
        m5 = m[:, None, None]                             # (N, 1, 1, T, V)
        y = torch.einsum('nkctv,kcvw->nctw', x * m5, A)   # messages from observed nodes only
        diag = torch.diagonal(A, dim1=-2, dim2=-1)        # (3, C', V): a missing node keeps its own (token) state
        y = y + torch.einsum('nkctw,kcw->nctw', x * (1 - m5), diag)
        tot = self.A_bar.sum(0)                           # (V,) full neighbourhood weight
        obs = torch.einsum('ntv,vw->ntw', m, self.A_bar) + torch.diagonal(self.A_bar) * (1 - m)
        y = y * (tot / obs.clamp(min=1e-3))[:, None]
        y = g.bn(y)
        y = y + g.down(xt)
        return g.relu(y)

MissingAwareGCN.forward = fp32_island(MissingAwareGCN.forward)

def propagate_mask(m):                                    # a node counts as informed if any 1-hop node was observed
    with torch.autocast(device_type='cuda', enabled=False):   # keep the mask in fp32
        return torch.clamp(torch.einsum('ntv,vw->ntw', m.float(), A_HOP.to(m.device)), max=1.0)

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None

class BdslNet(nn.Module):
    """n_ma = 0, masked_pool False, n_signers 0  ->  exactly the official SL-GCN forward pass (S0)."""
    def __init__(self, n_cls=60, in_ch=3, n_ma=0, masked_pool=False, n_signers=0, groups=16, block_size=41):
        super().__init__()
        self.base = SLGCN(num_class=n_cls, num_point=N_NODES, num_person=1, groups=groups, block_size=block_size,
                          graph='graph.bdsl49.Graph', graph_args={'labeling_mode': 'spatial'}, in_channels=in_ch)
        self.layers = [getattr(self.base, 'l%d' % i) for i in range(1, 11)]
        self.n_ma, self.masked_pool = n_ma, masked_pool
        chans = [in_ch, 64, 64, 64, 64, 128, 128, 128, 256, 256]
        self.ma = nn.ModuleList([MissingAwareGCN(self.layers[i].gcn1, chans[i]) for i in range(n_ma)])
        self.signer_head = (nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_signers))
                            if n_signers else None)

    def _unit(self, i, x, keep_prob, m):                  # official TCN_GCN_unit.forward, with an optional MA gcn
        u = self.layers[i]
        y = self.ma[i](x, m) if i < self.n_ma else u.gcn1(x)
        if u.attention:
            se = y.mean(-2); se1 = u.sigmoid(u.conv_sa(se)); y = y * se1.unsqueeze(-2) + y
            se = y.mean(-1); se1 = u.sigmoid(u.conv_ta(se)); y = y * se1.unsqueeze(-1) + y
            se = y.mean(-1).mean(-1); se1 = u.relu(u.fc1c(se)); se2 = u.sigmoid(u.fc2c(se1))
            y = y * se2.unsqueeze(-1).unsqueeze(-1) + y
        y = u.tcn1(y, keep_prob, u.A)
        x_skip = u.dropT_skip(u.dropSke(u.residual(x), keep_prob, u.A), keep_prob)
        return u.relu(y + x_skip)

    def forward(self, x, m=None, keep_prob=0.9, lam=0.0):  # x: (N, C, T, V), m: (N, T, V)
        N, C, T, V = x.shape
        b = self.base
        x = b.data_bn(x.permute(0, 3, 1, 2).reshape(N, V * C, T)).view(N, V, C, T).permute(0, 2, 3, 1).contiguous()
        m = torch.ones(N, T, V, device=x.device) if m is None else m.float()
        kps = [1.0] * 6 + [keep_prob] * 4                # the official schedule: DropGraph only in blocks 7-10
        for i in range(10):
            x = self._unit(i, x, kps[i], m)
            if i < self.n_ma:
                m = propagate_mask(m)
        if self.masked_pool:
            w = F.adaptive_max_pool2d(m[:, None], x.shape[2:])
            feat = (x * w).sum(dim=(2, 3)) / w.sum(dim=(2, 3)).clamp(min=1.0)
        else:
            feat = x.mean(dim=(2, 3))
        logits = b.fc(feat)
        if self.signer_head is not None and self.training and lam > 0:
            return logits, self.signer_head(GradReverse.apply(feat, lam))
        return logits, None

# ---- self-tests ----
torch.manual_seed(0)
_g = unit_gcn(3, 64, GRAPH_A, 16, N_NODES).to(device).eval()
_ma = MissingAwareGCN(_g, 3).to(device).eval()
_x = torch.randn(2, 3, 16, N_NODES, device=device)
with torch.no_grad():
    assert torch.allclose(_ma(_x, torch.ones(2, 16, N_NODES, device=device)), _g(_x), atol=1e-4), 'MA layer != SL-GCN when nothing is missing'
    _m = torch.ones(2, 16, N_NODES, device=device); _m[:, :, 7:28] = 0
    _a = _ma(_x, _m); _x2 = _x.clone(); _x2[:, :, :, 7:28] = 123.0
    assert torch.allclose(_a, _ma(_x2, _m), atol=1e-4), 'missing values leaked into the output'
_s0 = BdslNet().to(device).eval()
with torch.no_grad():
    _xx = torch.randn(2, 3, T_OUT, N_NODES, device=device)
    assert torch.allclose(_s0(_xx)[0], _s0.base(_xx[..., None], 1.0), atol=1e-4), 'wrapper != official SL-GCN forward'
if device == 'cuda':                                   # the overflow your teammate found, now contained
    _d = DropBlockT_1d(block_size=41).train()
    _h = torch.randn(64, 256, 16, N_NODES, device='cuda').half()
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        assert torch.isfinite(_d(_h, 0.9)).all(), 'DropBlockT still overflows under autocast'
print('self-tests passed: MA = SL-GCN when nothing is missing; missing values cannot leak; S0 = official forward;'
      ' DropBlock is finite under fp16 autocast')


device: cuda


/kaggle/working/CVPR21Chal-SLR/SL-GCN/model/decouple_gcn_attn.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.eyes = nn.Parameter(torch.tensor(torch.stack(


Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
self-tests passed: MA = SL-GCN when nothing is missing; missing values cannot leak; S0 = official forward; DropBlock is finite under fp16 autocast


In [10]:
# ---- Cell 10: dataset + training loop ----
USE_AMP = True     # mixed precision with fp32 islands (cell 9); cell 10b sets this to False if it is not clean
GRAD_CLIP = 5.0    # same for S0, S2, S3

class BdslDataset(torch.utils.data.Dataset):
    def __init__(self, X, M, y, idx, stream='joint', aug=False, signer_of=None):
        self.X, self.M, self.y, self.idx = X, M, y, np.asarray(idx)
        self.stream, self.aug, self.signer_of = stream, aug, signer_of
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        x, m = np.array(self.X[i]), np.array(self.M[i])
        if self.aug:
            x, m = augment(x, m, np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,)))))
        f, fm = make_stream(x, m, self.stream, use_mask=True)
        s = self.signer_of[i] if self.signer_of is not None else -1
        return torch.from_numpy(f), torch.from_numpy(fm.astype(np.float32)), int(self.y[i]), int(s)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    out, ys = [], []
    for x, m, y_, _ in loader:
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            out.append(model(x.to(device), m.to(device))[0].float().cpu())
        ys.append(y_)
    return torch.cat(out), torch.cat(ys)

def scores(logits, ys):
    p = logits.argmax(1)
    return dict(top1=(p == ys).float().mean().item(),
                top5=(logits.topk(5, 1).indices == ys[:, None]).any(1).float().mean().item(),
                f1=f1_score(ys.numpy(), p.numpy(), average='macro'))

def run(name, split, seed, n_ma=0, masked_pool=False, lam_max=0.0, stream='joint', epochs=60, lr=0.1, bs=64,
        warmup=5, tag='', eval_every=5):
    set_seed(seed)
    Xm = np.load(CACHE / 'X.npy', mmap_mode='r'); Mm = np.load(CACHE / 'M.npy', mmap_mode='r')
    tr_users = sorted(set(meta.user.values[split['train']]))
    sid = {u: k for k, u in enumerate(tr_users)}
    signer_of = np.array([sid.get(u, -1) for u in meta.user.values])
    tr = BdslDataset(Xm, Mm, y, split['train'], stream, aug=True, signer_of=signer_of)
    va = BdslDataset(Xm, Mm, y, split['val'], stream)
    tl = torch.utils.data.DataLoader(tr, batch_size=bs, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
    vl = torch.utils.data.DataLoader(va, batch_size=128, shuffle=False, num_workers=4)
    model = BdslNet(n_ma=n_ma, masked_pool=masked_pool, n_signers=len(tr_users) if lam_max > 0 else 0).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best = dict(f1=-1)
    evals = []                                   # (epoch, top1, f1) at every validation check
    steps_total, step = epochs * len(tl), 0
    print('%s %s seed %d | n_ma=%d masked_pool=%s lam_max=%.2f | %.2fM params | train %d val %d'
          % (name, tag, seed, n_ma, masked_pool, lam_max, n_params / 1e6, len(tr), len(va)))
    t_start = time.time()
    for ep in range(epochs):
        cur = lr * (ep + 1) / warmup if ep < warmup else 0.5 * lr * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for gp in opt.param_groups:
            gp['lr'] = cur
        keep_prob = max(0.9, 1.0 - 0.1 * ep / max(1, int(0.4 * epochs)))   # official ramp, scaled to the shorter run
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for x, m, y_, s_ in tl:
            x, m, y_, s_ = x.to(device), m.to(device), y_.to(device), s_.to(device)
            p = step / steps_total
            lam = lam_max * (2.0 / (1.0 + math.exp(-10 * p)) - 1.0)             # DANN schedule, 0 -> lam_max
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                logits, s_logits = model(x, m, keep_prob=keep_prob, lam=lam)
                loss = crit(logits, y_)
                if s_logits is not None:
                    loss = loss + F.cross_entropy(s_logits, s_)
            if not torch.isfinite(loss):
                raise RuntimeError('loss became %s at epoch %d - lower lr or lam_max' % (loss.item(), ep + 1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            scaler.step(opt); scaler.update()
            tot += loss.item() * len(y_); n += len(y_); step += 1
        if ep == 0:
            print('  first epoch took %.0f s -> about %.0f min for this run' % (time.time() - t0, (time.time() - t0) * epochs / 60))
        if (ep + 1) % eval_every == 0 or ep == epochs - 1:
            logits_v, ys_v = predict(model, vl)
            sc = scores(logits_v, ys_v)
            evals.append((ep + 1, sc['top1'], sc['f1']))
            if sc['f1'] > best['f1']:
                best = dict(sc, epoch=ep + 1)
            print('  ep %3d | loss %.3f | val top1 %.3f macroF1 %.3f | keep %.2f lam %.3f | %.0f min so far'
                  % (ep + 1, tot / n, sc['top1'], sc['f1'], keep_prob, lam, (time.time() - t_start) / 60))
    last = sc
    pred = pd.DataFrame(dict(idx=np.asarray(split['val']), pred=logits_v.argmax(1).numpy(), true=ys_v.numpy()))
    pred.to_csv(CACHE / ('pred_%s_%s_seed%d.csv' % (name, tag, seed)), index=False)   # last-epoch predictions
    torch.save(model.state_dict(), CACHE / ('%s_%s_seed%d_last.pt' % (name, tag, seed)))
    return dict(name=name, tag=tag, seed=seed, params_M=round(n_params / 1e6, 2), minutes=round((time.time() - t_start) / 60, 1),
                last_top1=round(last['top1'], 4), last_macroF1=round(last['f1'], 4),
                best_macroF1=round(best['f1'], 4), best_epoch=best['epoch'], epochs=epochs,
                last3_top1=round(float(np.mean([e[1] for e in evals[-3:]])), 4),
                last3_macroF1=round(float(np.mean([e[2] for e in evals[-3:]])), 4),
                eval_epochs=' '.join(str(e[0]) for e in evals))


In [11]:
# ---- Cell 10b: quick mixed-precision check (about 1 minute; session 1 showed it is clean) ----
def amp_check(n_steps=12):
    Xm = np.load(CACHE / 'X.npy', mmap_mode='r'); Mm = np.load(CACHE / 'M.npy', mmap_mode='r')
    ds = BdslDataset(Xm, Mm, y, inner[INNER_VAL_SIGNERS[0]]['train'], 'joint', aug=True)
    dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=True, num_workers=4, drop_last=True)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    bad = 0
    for kw in (dict(), dict(masked_pool=True), dict(n_ma=2, masked_pool=True)):
        set_seed(0)
        model = BdslNet(**kw).to(device).train()
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.SGD(params, lr=0.1, momentum=0.9, nesterov=True, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda')
        for k, (x, m, y_, _) in enumerate(dl):
            if k == n_steps:
                break
            x, m, y_ = x.to(device), m.to(device), y_.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits, _ = model(x, m, keep_prob=0.9)
                loss = crit(logits, y_)
            if not torch.isfinite(loss) or not torch.isfinite(logits).all():
                bad += 1
                continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
            scaler.step(opt); scaler.update()
        del model, opt
        torch.cuda.empty_cache()
    return bad

if device == 'cuda':
    n_bad = amp_check()
    USE_AMP = n_bad == 0
    print('non-finite steps: %d -> USE_AMP = %s' % (n_bad, USE_AMP))
    if not USE_AMP:
        print('WARNING: session 1 used mixed precision; fp32 results would not be comparable and are ~1.3x slower. Send me this output.')
else:
    USE_AMP = False


Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!


/kaggle/working/CVPR21Chal-SLR/SL-GCN/model/decouple_gcn_attn.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.eyes = nn.Parameter(torch.tensor(torch.stack(


Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
non-finite steps: 0 -> USE_AMP = True


In [12]:
# ---- Cell 11: session-2 jobs (resumable, clock-guarded) ----
EPOCHS = 38               # frozen: must match seed 0 from session 1
TIME_BUDGET_H = 10.5      # Kaggle stops at 12 h; keep a margin
CFG = {'S0_slgcn': dict(n_ma=0, masked_pool=False, lam_max=0.0),
       'S1_masked_pool': dict(n_ma=0, masked_pool=True, lam_max=0.0),   # ablation: pooling only
       'S2_missing_aware': dict(n_ma=2, masked_pool=True, lam_max=0.0),
       'S3_ma_signer_adv': dict(n_ma=2, masked_pool=True, lam_max=0.1)}
# priority order: seed 1 of S0/S2 fold by fold (each fold is a complete pair), then the S1 ablation
JOBS = ([(1, n, v) for v in INNER_VAL_SIGNERS for n in ('S0_slgcn', 'S2_missing_aware')]
        + [(0, 'S1_masked_pool', v) for v in INNER_VAL_SIGNERS])
# later sessions: seed 2 -> [(2, n, v) for v in INNER_VAL_SIGNERS for n in ('S0_slgcn', 'S2_missing_aware')]

res_path = CACHE / 'week2_inner_results.csv'
if not res_path.exists():                     # bring in session-1 results (seed 0) and its predictions
    prev = sorted(INPUT_ROOT.rglob('week2_inner_results.csv'), key=lambda p: p.stat().st_mtime)
    if prev:
        old = pd.read_csv(prev[-1])
        old = old[(old.amp == USE_AMP) & (old.epochs == EPOCHS)]
        old.to_csv(res_path, index=False)
        n_pred = 0
        for f in prev[-1].parent.glob('pred_*.csv'):
            shutil.copy(f, CACHE / f.name); n_pred += 1
        print('resumed %d finished runs and %d prediction files from %s' % (len(old), n_pred, prev[-1]))
    else:
        print('WARNING: no session-1 results found. Add the session-1 notebook output as an input, '
              'otherwise this session cannot be compared with seed 0.')
done = pd.read_csv(res_path) if res_path.exists() else pd.DataFrame(columns=['name', 'tag', 'seed', 'epochs', 'minutes'])
results = done.to_dict('records')

def expected_minutes(name):                   # real run time from earlier runs of this model, else the slowest seen
    seen = done[done.name == name].minutes if len(done) else []
    if len(seen):
        return float(np.max(seen))
    return float(done.minutes.max()) if len(done) else 75.0

print('jobs this session:')
for seed, name, v in JOBS:
    print('   seed %d  %-18s U%d  (~%.0f min)' % (seed, name, v, expected_minutes(name)))
for seed, name, v in JOBS:
    tag = 'U%d' % v
    if len(done) and ((done.name == name) & (done.tag == tag) & (done.seed == seed) & (done.epochs == EPOCHS)).any():
        continue
    used_h = (time.time() - SESSION_START) / 3600
    need_h = expected_minutes(name) * 1.05 / 60
    if used_h + need_h > TIME_BUDGET_H:
        print('STOP before %s %s seed %d: %.1f h used + %.1f h needed > %.1f h budget. Saved results are kept.'
              % (name, tag, seed, used_h, need_h, TIME_BUDGET_H))
        break
    c = CFG[name]
    r = run(name, inner[v], seed, c['n_ma'], c['masked_pool'], c['lam_max'], epochs=EPOCHS, tag=tag)
    r['val_signer'] = v
    r['amp'] = USE_AMP
    results.append(r)
    pd.DataFrame(results).to_csv(res_path, index=False)
    done = pd.DataFrame(results)
print('session time used: %.1f h' % ((time.time() - SESSION_START) / 3600))
res_df = pd.DataFrame(results)
res_df


resumed 9 finished runs and 9 prediction files from /kaggle/input/notebooks/sumaiya110/session-1-notebook/bdslw60_w2/week2_inner_results.csv
jobs this session:
   seed 1  S0_slgcn           U1  (~68 min)
   seed 1  S2_missing_aware   U1  (~72 min)
   seed 1  S0_slgcn           U10  (~68 min)
   seed 1  S2_missing_aware   U10  (~72 min)
   seed 1  S0_slgcn           U13  (~68 min)
   seed 1  S2_missing_aware   U13  (~72 min)
   seed 0  S1_masked_pool     U1  (~72 min)
   seed 0  S1_masked_pool     U10  (~72 min)
   seed 0  S1_masked_pool     U13  (~72 min)
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
S0_slgcn U1 seed 1 | n_ma=0 masked_pool=False lam_max=0.00 | 4.63M params | train 7054 val 969


/kaggle/working/CVPR21Chal-SLR/SL-GCN/model/decouple_gcn_attn.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.eyes = nn.Parameter(torch.tensor(torch.stack(


  first epoch took 88 s -> about 56 min for this run
  ep   5 | loss 1.392 | val top1 0.598 macroF1 0.567 | keep 0.97 lam 0.000 | 8 min so far
  ep  10 | loss 1.107 | val top1 0.710 macroF1 0.675 | keep 0.94 lam 0.000 | 16 min so far
  ep  15 | loss 1.048 | val top1 0.714 macroF1 0.691 | keep 0.91 lam 0.000 | 25 min so far
  ep  20 | loss 0.983 | val top1 0.751 macroF1 0.728 | keep 0.90 lam 0.000 | 33 min so far
  ep  25 | loss 0.924 | val top1 0.783 macroF1 0.767 | keep 0.90 lam 0.000 | 41 min so far
  ep  30 | loss 0.891 | val top1 0.779 macroF1 0.759 | keep 0.90 lam 0.000 | 50 min so far
  ep  35 | loss 0.880 | val top1 0.783 macroF1 0.760 | keep 0.90 lam 0.000 | 58 min so far
  ep  38 | loss 0.878 | val top1 0.794 macroF1 0.776 | keep 0.90 lam 0.000 | 63 min so far
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
S2_missing_aware U1 seed 1 | n

/kaggle/working/CVPR21Chal-SLR/SL-GCN/model/decouple_gcn_attn.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.eyes = nn.Parameter(torch.tensor(torch.stack(


  first epoch took 95 s -> about 60 min for this run
  ep   5 | loss 1.407 | val top1 0.608 macroF1 0.582 | keep 0.97 lam 0.000 | 9 min so far
  ep  10 | loss 1.078 | val top1 0.697 macroF1 0.674 | keep 0.94 lam 0.000 | 17 min so far
  ep  15 | loss 1.022 | val top1 0.723 macroF1 0.693 | keep 0.91 lam 0.000 | 26 min so far
  ep  20 | loss 0.956 | val top1 0.794 macroF1 0.773 | keep 0.90 lam 0.000 | 35 min so far
  ep  25 | loss 0.910 | val top1 0.788 macroF1 0.763 | keep 0.90 lam 0.000 | 44 min so far
  ep  30 | loss 0.888 | val top1 0.820 macroF1 0.795 | keep 0.90 lam 0.000 | 52 min so far
  ep  35 | loss 0.874 | val top1 0.791 macroF1 0.766 | keep 0.90 lam 0.000 | 61 min so far
  ep  38 | loss 0.870 | val top1 0.813 macroF1 0.789 | keep 0.90 lam 0.000 | 66 min so far
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
S0_slgcn U10 seed 1 | n_ma=0 m

/kaggle/working/CVPR21Chal-SLR/SL-GCN/model/decouple_gcn_attn.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.eyes = nn.Parameter(torch.tensor(torch.stack(


... (10 outputs left out)

  first epoch took 97 s -> about 61 min for this run
  ep   5 | loss 1.423 | val top1 0.710 macroF1 0.524 | keep 0.97 lam 0.000 | 9 min so far
  ep  10 | loss 1.097 | val top1 0.830 macroF1 0.723 | keep 0.94 lam 0.000 | 18 min so far
  ep  15 | loss 1.021 | val top1 0.911 macroF1 0.812 | keep 0.91 lam 0.000 | 27 min so far
  ep  20 | loss 0.959 | val top1 0.924 macroF1 0.816 | keep 0.90 lam 0.000 | 36 min so far
  ep  25 | loss 0.909 | val top1 0.924 macroF1 0.777 | keep 0.90 lam 0.000 | 45 min so far
  ep  30 | loss 0.873 | val top1 0.940 macroF1 0.769 | keep 0.90 lam 0.000 | 53 min so far
  ep  35 | loss 0.860 | val top1 0.940 macroF1 0.810 | keep 0.90 lam 0.000 | 62 min so far
  ep  38 | loss 0.859 | val top1 0.940 macroF1 0.786 | keep 0.90 lam 0.000 | 68 min so far
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
Attention Enabled!
S1_masked_pool U13 seed 0 | n_

/kaggle/working/CVPR21Chal-SLR/SL-GCN/model/decouple_gcn_attn.py:118: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.eyes = nn.Parameter(torch.tensor(torch.stack(


  first epoch took 94 s -> about 59 min for this run
  ep   5 | loss 1.371 | val top1 0.644 macroF1 0.597 | keep 0.97 lam 0.000 | 9 min so far
  ep  10 | loss 1.079 | val top1 0.688 macroF1 0.640 | keep 0.94 lam 0.000 | 17 min so far
  ep  15 | loss 1.023 | val top1 0.742 macroF1 0.716 | keep 0.91 lam 0.000 | 26 min so far
  ep  20 | loss 0.973 | val top1 0.818 macroF1 0.786 | keep 0.90 lam 0.000 | 35 min so far
  ep  25 | loss 0.918 | val top1 0.847 macroF1 0.819 | keep 0.90 lam 0.000 | 43 min so far
  ep  30 | loss 0.877 | val top1 0.826 macroF1 0.797 | keep 0.90 lam 0.000 | 52 min so far
  ep  35 | loss 0.869 | val top1 0.841 macroF1 0.814 | keep 0.90 lam 0.000 | 61 min so far
  ep  38 | loss 0.869 | val top1 0.845 macroF1 0.822 | keep 0.90 lam 0.000 | 66 min so far
session time used: 10.0 h


,name,tag,seed,params_M,minutes,last_top1,last_macroF1,best_macroF1,best_epoch,epochs,val_signer,amp,last3_top1,last3_macroF1,eval_epochs
0,S0_slgcn,U1,0,4.63,63.0,0.7792,0.7585,0.7793,25,38,1,True,NaN,NaN,NaN
1,S2_missing_aware,U1,0,4.63,66.7,0.8008,0.7811,0.7931,30,38,1,True,NaN,NaN,NaN
2,S3_ma_signer_adv,U1,0,4.67,66.7,0.8277,0.8094,0.8094,38,38,1,True,NaN,NaN,NaN
3,S0_slgcn,U10,0,4.63,67.8,0.9243,0.7960,0.8499,35,38,10,True,NaN,NaN,NaN
4,S2_missing_aware,U10,0,4.63,71.6,0.9138,0.8140,0.8140,38,38,10,True,NaN,NaN,NaN
5,S3_ma_signer_adv,U10,0,4.67,71.6,0.8956,0.7827,0.7827,38,38,10,True,NaN,NaN,NaN
6,S0_slgcn,U13,0,4.63,65.6,0.8406,0.8120,0.8120,38,38,13,True,NaN,NaN,NaN
7,S2_missing_aware,U13,0,4.63,69.4,0.8680,0.8416,0.8416,38,38,13,True,NaN,NaN,NaN
8,S3_ma_signer_adv,U13,0,4.67,69.3,0.8551,0.8266,0.8345,30,38,13,True,NaN,NaN,NaN
9,S0_slgcn,U1,1,4.63,62.9,0.7936,0.7761,0.7761,38,38,1,True,0.7853,0.7652,5 10 15 20 25 30 35 38


In [13]:
# ---- Cell 12: analysis (primary score: mean macro-F1 of the last 3 validation checks) ----
# Seed-0 last-3 values were read from the session-1 log (epochs 30, 35, 38); later runs record them directly.
SEED0_LAST3 = {
 "S0_slgcn|U1": {
  "last3_top1": 0.7823,
  "last3_macroF1": 0.762
 },
 "S2_missing_aware|U1": {
  "last3_top1": 0.8063,
  "last3_macroF1": 0.7867
 },
 "S3_ma_signer_adv|U1": {
  "last3_top1": 0.822,
  "last3_macroF1": 0.8037
 },
 "S0_slgcn|U10": {
  "last3_top1": 0.928,
  "last3_macroF1": 0.8147
 },
 "S2_missing_aware|U10": {
  "last3_top1": 0.92,
  "last3_macroF1": 0.8107
 },
 "S3_ma_signer_adv|U10": {
  "last3_top1": 0.8967,
  "last3_macroF1": 0.7727
 },
 "S0_slgcn|U13": {
  "last3_top1": 0.8263,
  "last3_macroF1": 0.7967
 },
 "S2_missing_aware|U13": {
  "last3_top1": 0.8593,
  "last3_macroF1": 0.8337
 },
 "S3_ma_signer_adv|U13": {
  "last3_top1": 0.852,
  "last3_macroF1": 0.8233
 }
}
res_df = pd.read_csv(CACHE / 'week2_inner_results.csv')
for col in ('last3_top1', 'last3_macroF1'):
    if col not in res_df.columns:
        res_df[col] = np.nan
for i, r in res_df.iterrows():
    key = '%s|%s' % (r['name'], r['tag'])
    if r['seed'] == 0 and pd.isna(r['last3_macroF1']) and key in SEED0_LAST3:
        res_df.loc[i, 'last3_top1'] = SEED0_LAST3[key]['last3_top1']
        res_df.loc[i, 'last3_macroF1'] = SEED0_LAST3[key]['last3_macroF1']
res_df.to_csv(CACHE / 'week2_inner_results_with_last3.csv', index=False)

print('runs per model and seed:')
print(res_df.groupby(['name', 'seed']).size().unstack(fill_value=0).to_string())
print('\nmean over all runs:')
print(res_df.groupby('name')[['last3_macroF1', 'last3_top1', 'last_macroF1', 'best_macroF1', 'minutes']]
      .agg(['mean', 'std', 'count']).round(4).to_string())

def paired(a, b, col='last3_macroF1'):
    pv = res_df.pivot_table(index=['val_signer', 'seed'], columns='name', values=col)
    if a not in pv or b not in pv:
        return
    d = (pv[a] - pv[b]).dropna()
    if len(d) == 0:
        return
    t = d.mean() / (d.std() / np.sqrt(len(d))) if len(d) > 1 and d.std() > 0 else float('nan')
    print('%-18s - %-18s %s: mean %+.4f | std %.4f | mean/SE %.2f | better in %d of %d pairs | per signer %s'
          % (a, b, col, d.mean(), d.std() if len(d) > 1 else float('nan'), t, int((d > 0).sum()), len(d),
             d.groupby(level=0).mean().round(4).to_dict()))
print('\npaired differences (same fold and seed):')
for a, b in (('S2_missing_aware', 'S0_slgcn'), ('S1_masked_pool', 'S0_slgcn'), ('S2_missing_aware', 'S1_masked_pool'),
             ('S3_ma_signer_adv', 'S0_slgcn')):
    paired(a, b)
    paired(a, b, 'last_macroF1')

# last-epoch predictions: by hand and by missing share, the missing share split WITHIN each signer
preds = []
for f in sorted(CACHE.glob('pred_*.csv')):
    mt = re.match(r'pred_(.+)_(U\d+)_seed(\d+)\.csv', f.name)
    if mt:
        preds.append(pd.read_csv(f).assign(name=mt[1], val_signer=int(mt[2][1:]), seed=int(mt[3])))
if preds:
    P = pd.concat(preds).join(meta[['hand', 'hand_missing']], on='idx')
    P['correct'] = (P.pred == P.true).astype(float)
    print('\naccuracy by hand dominance (mean over seeds, last epoch):')
    print(P.pivot_table(index=['val_signer', 'hand'], columns='name', values='correct', aggfunc='mean').round(3).to_string())
    # terciles of the missing share computed separately for each signer, so signers are not mixed across bins
    P['miss_tercile'] = P.groupby('val_signer').hand_missing.transform(
        lambda s: pd.qcut(s.rank(method='first'), 3, labels=['low', 'mid', 'high']).astype(str))
    tab = P.pivot_table(index=['val_signer', 'miss_tercile'], columns='name', values='correct', aggfunc='mean')
    print('\naccuracy by missing-hand tercile WITHIN each signer:')
    print(tab.round(3).to_string())
    print('\ngain over S0 by within-signer tercile (mean over signers):')
    for other in [c for c in tab.columns if c != 'S0_slgcn']:
        g = (tab[other] - tab['S0_slgcn']).groupby(level=1).mean().reindex(['low', 'mid', 'high'])
        print('   %-18s %s' % (other, g.round(3).to_dict()))
    print('(if the missing-aware convolution is what helps, S2 - S0 should grow from low to high, and S2 should beat S1 most in "high")')


runs per model and seed:
seed              0  1
name                  
S0_slgcn          3  3
S1_masked_pool    3  0
S2_missing_aware  3  3
S3_ma_signer_adv  3  0

mean over all runs:
                 last3_macroF1               last3_top1               last_macroF1               best_macroF1               minutes              
                          mean     std count       mean     std count         mean     std count         mean     std count    mean     std count
name                                                                                                                                             
S0_slgcn                0.7976  0.0348     6     0.8501  0.0610     6       0.7978  0.0355     6       0.8139  0.0350     6   65.40  2.1288     6
S1_masked_pool          0.7945  0.0143     3     0.8605  0.0707     3       0.7996  0.0194     3       0.8098  0.0160     3   65.50  2.3065     3
S2_missing_aware        0.8057  0.0206     6     0.8573  0.0460     6       0.8102  0.